<h2>Apri questo notebook in Google Colab</h2>

<div align="left" style="margin: 20px 0;">
  <a href="https://colab.research.google.com/github/LeonardoCofone/The-AI-Handbook/blob/main/chapter_5_Machine_Learning.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

# Capitolo 5, Regressione

In questo notebook costruiamo la regressione lineare da zero, pezzo per pezzo.  
Partiamo dal modello più semplice possibile (una retta) e arriviamo alla discesa del gradiente, l'algoritmo di ottimizzazione alla base di quasi tutto il Machine Learning.

Tutto su dati sintetici: così puoi concentrarti sui concetti senza distrarti con la pulizia del dataset.

## 0. Setup, importare le librerie

In [ ]:
%pip install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, SGDRegressor

np.random.seed(42)
print('Setup completato!')

## 1. Il modello: f(x) = wx + b

Simuliamo un problema classico: prevedere il prezzo di un appartamento in base alla sua superficie (mq).  
Il prezzo reale segue la relazione `prezzo = 200 * mq + 50000`, con un po' di rumore.

In [ ]:
# Dati sintetici: 100 appartamenti
m = 100
X = 2 * np.random.rand(m, 1) * 50 + 30   # superfici tra 30 e 130 mq
y = 200 * X + 50_000 + np.random.randn(m, 1) * 5_000  # prezzo con rumore

plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.6, s=20)
plt.xlabel('Superficie (mq)')
plt.ylabel('Prezzo (€)')
plt.title('Dataset: prezzo appartamenti')
plt.tight_layout()
plt.show()

In [ ]:
# Il ruolo di w e b: prova a cambiare questi valori e guarda come cambia la retta
w = 200       # pendenza: ogni mq in più vale 200€
b = 50_000    # intercetta: il prezzo base a 0 mq

x_line = np.linspace(X.min(), X.max(), 100)
y_line = w * x_line + b

plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.6, s=20, label='Dati reali')
plt.plot(x_line, y_line, 'r-', linewidth=2, label=f'f(x) = {w}x + {b:,}')
plt.xlabel('Superficie (mq)')
plt.ylabel('Prezzo (€)')
plt.title('Modello lineare: f(x) = wx + b')
plt.legend()
plt.tight_layout()
plt.show()

# Una previsione manuale
x_nuovo = 100   # appartamento di 100 mq
y_hat = w * x_nuovo + b
print(f'Previsione per {x_nuovo} mq: {y_hat:,.0f}€')

---
## 2. La funzione di costo J(w, b)

Come misuriamo quanto bene (o male) sta andando il modello durante il training?  
Usiamo la funzione di costo MSE, con una piccola variazione: si divide per `2m` invece di `m`.  
Non cambia il punto di minimo, ma semplifica il calcolo della derivata.

In [ ]:
def compute_cost(X, y, w, b):
    """J(w,b) = (1/2m) * sum((wx+b - y)²)"""
    m = len(y)
    predictions = w * X + b
    errors = predictions - y
    return (1 / (2 * m)) * np.sum(errors ** 2)

# Con i parametri giusti il costo è basso
J_buono  = compute_cost(X, y, w=200, b=50_000)
J_sbagliato = compute_cost(X, y, w=100, b=10_000)

print(f'J con w=200, b=50000 (parametri giusti):  {J_buono:,.0f}')
print(f'J con w=100, b=10000 (parametri sbagliati): {J_sbagliato:,.0f}')

In [ ]:
# Visualizziamo J al variare di w (fissando b al valore ottimale)
# → Parabola con un solo minimo: la forma che rende la discesa del gradiente affidabile

w_range = np.linspace(0, 400, 200)
J_values = [compute_cost(X, y, w=wi, b=50_000) for wi in w_range]

plt.figure(figsize=(7, 4))
plt.plot(w_range, J_values)
plt.axvline(x=200, color='red', linestyle='--', alpha=0.7, label='w ottimale = 200')
plt.xlabel('w (peso)')
plt.ylabel('J(w)')
plt.title('Funzione di costo J al variare di w  →  parabola con un solo minimo')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Con entrambi i parametri: superficie 3D a forma di ciotola
w_vals = np.linspace(50, 350, 60)
b_vals = np.linspace(20_000, 80_000, 60)
W, B   = np.meshgrid(w_vals, b_vals)
J_grid = np.array([[compute_cost(X, y, wi, bi) for wi in w_vals] for bi in b_vals])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Superficie 3D
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(W, B, J_grid, cmap='viridis', alpha=0.8)
ax1.set_xlabel('w')
ax1.set_ylabel('b')
ax1.set_zlabel('J')
ax1.set_title('Superficie J(w, b)  →  ciotola')

# Contour plot
ax2 = plt.subplot(122)
cp = ax2.contourf(W, B, J_grid, levels=30, cmap='viridis')
ax2.plot(200, 50_000, 'r*', markersize=15, label='Minimo globale')
plt.colorbar(cp, ax=ax2)
ax2.set_xlabel('w')
ax2.set_ylabel('b')
ax2.set_title('Contour plot: cerchi concentrici, centro = minimo')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 3. Discesa del Gradiente

La discesa del gradiente minimizza J aggiornando w e b ad ogni iterazione nella direzione di discesa più ripida.

**Regola fondamentale**: w e b vanno aggiornati *contemporaneamente*, usando i vecchi valori per calcolare entrambi gli aggiornamenti prima di applicarli.

In [ ]:
# Aggiornamento SBAGLIATO: si usa il nuovo w per calcolare l'aggiornamento di b
# w = w - alpha * dw   ← w già aggiornato
# b = b - alpha * db   ← db calcolato col nuovo w → ERRORE

# Aggiornamento CORRETTO: si calcolano entrambi i nuovi valori, poi si applicano insieme
# temp_w = w - alpha * dw
# temp_b = b - alpha * db
# w = temp_w           ← si applica solo dopo
# b = temp_b           ← si applica solo dopo

def gradient_descent(X, y, learning_rate=0.0001, n_iterations=1000):
    m = len(y)
    w, b = 0.0, 0.0           # inizializzazione a zero
    cost_history = []

    for _ in range(n_iterations):
        predictions = w * X + b
        errors = predictions - y

        # Derivate parziali
        dw = (1 / m) * np.sum(errors * X)
        db = (1 / m) * np.sum(errors)

        # Aggiornamento simultaneo (CORRETTO)
        temp_w = w - learning_rate * dw
        temp_b = b - learning_rate * db
        w, b = temp_w, temp_b

        cost_history.append(compute_cost(X, y, w, b))

    return w, b, cost_history

w_fit, b_fit, history = gradient_descent(X, y, learning_rate=0.0001, n_iterations=2000)
print(f'w trovato: {w_fit:.1f}   (vero: 200)')
print(f'b trovato: {b_fit:.0f}  (vero: 50000)')

In [ ]:
# Curva di apprendimento: J deve scendere monotonicamente
plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel('Iterazione')
plt.ylabel('J(w, b)')
plt.title('Discesa del Gradiente: J scende ad ogni iterazione')
plt.tight_layout()
plt.show()

# Retta trovata vs dati
x_line = np.linspace(X.min(), X.max(), 100)
plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.5, s=20, label='Dati reali')
plt.plot(x_line, w_fit * x_line + b_fit, 'r-', linewidth=2, label='Retta trovata dal GD')
plt.xlabel('Superficie (mq)')
plt.ylabel('Prezzo (€)')
plt.title('Risultato della Discesa del Gradiente')
plt.legend()
plt.tight_layout()
plt.show()

### 3.1 Il Learning Rate α

Il learning rate è l'iperparametro più importante della discesa del gradiente.  
Troppo piccolo → lentissimo. Troppo grande → diverge. Giusto → converge rapido e stabile.

In [ ]:
learning_rates = {
    'Troppo piccolo (1e-6)': 1e-6,
    'Giusto (1e-4)':         1e-4,
    'Troppo grande (5e-3)':  5e-3,
}

plt.figure(figsize=(10, 4))

for label, lr in learning_rates.items():
    _, _, hist = gradient_descent(X, y, learning_rate=lr, n_iterations=500)
    # Clipping per non far esplodere il grafico con il learning rate troppo grande
    hist_clipped = np.clip(hist, 0, hist[0] * 2)
    plt.plot(hist_clipped, label=label)

plt.xlabel('Iterazione')
plt.ylabel('J(w, b)')
plt.title('Effetto del Learning Rate α')
plt.legend()
plt.tight_layout()
plt.show()

# Verifica: con α troppo grande J invece di scendere oscilla o sale
_, _, hist_grande = gradient_descent(X, y, learning_rate=5e-3, n_iterations=10)
print('J con α troppo grande (prime 10 iterazioni):')
for i, j in enumerate(hist_grande):
    print(f'  iter {i+1}: J = {j:.2e}')

---
## 4. Batch GD vs Stochastic GD vs Mini-Batch GD

Tre varianti della discesa del gradiente con caratteristiche diverse.  
Per la regressione lineare tutte e tre trovano la soluzione: le differenze diventano importanti con modelli più complessi.

In [ ]:
def batch_gd(X, y, lr=1e-4, n_iter=200):
    """Batch GD: usa tutti gli m esempi ad ogni aggiornamento."""
    m = len(y)
    w, b = 0.0, 0.0
    history = []
    for _ in range(n_iter):
        errors = (w * X + b) - y
        w -= lr * (1/m) * np.sum(errors * X)
        b -= lr * (1/m) * np.sum(errors)
        history.append(compute_cost(X, y, w, b))
    return w, b, history

def stochastic_gd(X, y, lr=1e-4, n_epochs=200):
    """SGD: usa 1 solo esempio scelto casualmente ad ogni aggiornamento."""
    m = len(y)
    w, b = 0.0, 0.0
    history = []
    for _ in range(n_epochs):
        idx = np.random.randint(m)
        xi, yi = X[idx:idx+1], y[idx:idx+1]
        error = (w * xi + b) - yi
        w -= lr * np.sum(error * xi)
        b -= lr * np.sum(error)
        history.append(compute_cost(X, y, w, b))
    return w, b, history

def minibatch_gd(X, y, lr=1e-4, n_epochs=200, batch_size=32):
    """Mini-Batch GD: usa k esempi casuali ad ogni aggiornamento."""
    m = len(y)
    w, b = 0.0, 0.0
    history = []
    for _ in range(n_epochs):
        indices = np.random.choice(m, batch_size, replace=False)
        xi, yi = X[indices], y[indices]
        error = (w * xi + b) - yi
        w -= lr * (1/batch_size) * np.sum(error * xi)
        b -= lr * (1/batch_size) * np.sum(error)
        history.append(compute_cost(X, y, w, b))
    return w, b, history

_, _, hist_batch     = batch_gd(X, y)
_, _, hist_sgd       = stochastic_gd(X, y)
_, _, hist_minibatch = minibatch_gd(X, y)

plt.figure(figsize=(10, 4))
plt.plot(hist_batch,     label='Batch GD     — scesa regolare',  linewidth=2)
plt.plot(hist_minibatch, label='Mini-Batch GD — compromesso',     linewidth=1.5, alpha=0.85)
plt.plot(hist_sgd,       label='Stochastic GD — molto rumoroso', linewidth=1,   alpha=0.7)
plt.xlabel('Iterazione / Epoca')
plt.ylabel('J(w, b)')
plt.title('Batch GD vs Mini-Batch GD vs Stochastic GD')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Con scikit-learn: SGDRegressor fa esattamente lo stesso con una sola riga
from sklearn.preprocessing import StandardScaler

# SGDRegressor vuole le feature scalate
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

sgd_reg = SGDRegressor(max_iter=1000, tol=1e-5, penalty=None,
                       eta0=0.01, n_iter_no_change=100, random_state=42)
sgd_reg.fit(X_scaled, y.ravel())

print(f'SGDRegressor — w: {sgd_reg.coef_[0]:.1f},  b: {sgd_reg.intercept_[0]:.0f}')

---
## 5. Normal Equation

Alternativa alla discesa del gradiente: calcola i parametri ottimali **in un solo passaggio**, con una formula chiusa.  
Non serve learning rate, non serve convergenza. Ma non scala bene con molte feature.

In [ ]:
# Aggiungiamo una colonna di 1 per il bias (intercetta)
X_b = np.c_[np.ones((m, 1)), X]

# Normal Equation: θ = (XᵀX)⁻¹ Xᵀ y
theta_ne = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print(f'Normal Equation     — b: {theta_ne[0,0]:.0f},  w: {theta_ne[1,0]:.1f}')

# Versione più stabile: pseudo-inversa (preferita in pratica)
theta_pinv = np.linalg.pinv(X_b) @ y
print(f'Pseudo-inversa      — b: {theta_pinv[0,0]:.0f},  w: {theta_pinv[1,0]:.1f}')

# Confronto con scikit-learn LinearRegression (usa SVD internamente, stessa idea)
lr_model = LinearRegression()
lr_model.fit(X, y)
print(f'LinearRegression    — b: {lr_model.intercept_[0]:.0f},  w: {lr_model.coef_[0,0]:.1f}')

print(f'\nValore vero         — b: 50000,  w: 200.0')

In [ ]:
# Confronto di velocità: Normal Equation vs Gradient Descent al crescere delle feature
# (simuliamo dataset con n feature crescenti)
import time

n_features_list = [10, 100, 500, 1000, 3000]
times_ne, times_gd = [], []

for n in n_features_list:
    X_test = np.random.rand(500, n)
    y_test = np.random.rand(500, 1)
    X_b_test = np.c_[np.ones((500, 1)), X_test]

    # Normal Equation
    t0 = time.time()
    _ = np.linalg.pinv(X_b_test) @ y_test
    times_ne.append(time.time() - t0)

    # Gradient Descent (100 iterazioni)
    t0 = time.time()
    model = LinearRegression()
    model.fit(X_test, y_test)
    times_gd.append(time.time() - t0)

plt.figure(figsize=(7, 4))
plt.plot(n_features_list, times_ne, 'o-', label='Normal Equation  O(n³)')
plt.plot(n_features_list, times_gd, 's--', label='LinearRegression (SVD)')
plt.xlabel('Numero di feature')
plt.ylabel('Tempo (s)')
plt.title('Normal Equation vs GD: scalabilità')
plt.legend()
plt.tight_layout()
plt.show()
print('Con poche feature la Normal Equation è velocissima.')
print('Con molte feature la complessità O(n³) la rende proibitiva.')

---
## Riepilogo: quando usare cosa?

| Metodo | Quando usarlo |
|---|---|
| **Normal Equation** | Dataset piccoli, poche feature, vuoi la soluzione esatta senza iperparametri |
| **Batch GD** | Dataset medio, convergenza stabile, hai tempo |
| **Mini-Batch GD** | Default nella pratica, soprattutto per il Deep Learning |
| **Stochastic GD** | Dataset molto grandi, vuoi aggiornamenti frequenti |
| **`LinearRegression`** (sklearn) | Sempre, per la regressione lineare in produzione — gestisce tutto internamente |

Nel prossimo capitolo: **regressione multipla** — stessa idea, ma con più feature.